# Phase 3.3 Step 4: Production MD (Kaggle GPU Version)
**Steps covered:**
- Step 19: 10 ns production simulation at 310 K, 1 bar (PME 1.2 nm)
- Step 20: GPU acceleration (-nb gpu -pme gpu)
- Step 21: Checkpointing every 1 ns

**Runtime:** T4 GPU → ~1-2 hours for 10 ns

**All paths fixed for Kaggle (/kaggle/working/ instead of /content/)**

In [17]:
# Cell 0 — Install GROMACS + check GPU
import subprocess, zipfile, glob, re, os, shutil
from pathlib import Path

W = '/kaggle/working/'  # all files go here
os.chdir(W)

# Check GPU
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0:
    print(f'✓ GPU: {gpu.stdout.strip()}')
else:
    print('⚠ No GPU — switch to GPU T4 x2 in Session options')

# Install GROMACS
print('\nInstalling GROMACS...')
subprocess.run(['apt-get', 'install', '-y', 'gromacs'], capture_output=True)
ver = subprocess.run(['gmx', '--version'], capture_output=True, text=True)
if ver.returncode == 0:
    lines = [l for l in ver.stdout.splitlines() if 'GROMACS version' in l]
    print(f'✓ {lines[0].strip()}' if lines else '✓ GROMACS installed')
else:
    raise RuntimeError('GROMACS install failed')

GMX = 'gmx'
print(f'✓ GMX = "{GMX}"')
print(f'✓ Working dir: {os.getcwd()}')

✓ GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

Installing GROMACS...
✓ GROMACS version:    2021.4-Ubuntu-2021.4-2
✓ GMX = "gmx"
✓ Working dir: /kaggle/working


In [18]:
# Cell 1 — Copy files from Kaggle dataset to working directory
# Dataset must be attached as: insr-md-files
# Contains: npt2.gro, npt2.cpt, topol.top, posre.itp, gromacs zip

input_base = '/kaggle/input/'
W = '/kaggle/working/'

for f in Path(input_base).rglob('*'):
    if f.is_file():
        dest = Path(W) / f.name
        shutil.copy(str(f), str(dest))
        print(f'✓ Copied {f.name} ({f.stat().st_size/1024:.1f} KB)')

print('\nFinal check:')
for fname in ['npt2.gro', 'npt2.cpt', 'topol.top', 'posre.itp']:
    status = '✓' if Path(f'{W}{fname}').exists() else '✗'
    print(f'  {status} {fname}')

✓ Copied topol.top (1134.0 KB)
✓ Copied npt2.cpt (962.6 KB)
✓ Copied npt2.gro (2762.0 KB)
✓ Copied posre.itp (172.9 KB)
✓ Copied topol.top (1134.0 KB)
✓ Copied candidate_0004.gro (0.8 KB)
✓ Copied ions.mdp (0.1 KB)
✓ Copied solvated.gro (1808.6 KB)
✓ Copied protein.gro (180.9 KB)
✓ Copied system.gro (1801.3 KB)
✓ Copied candidate_0004.sdf (1.8 KB)
✓ Copied complex.top (1134.0 KB)
✓ Copied box.gro (180.9 KB)
✓ Copied complex.gro (181.6 KB)

Final check:
  ✓ npt2.gro
  ✓ npt2.cpt
  ✓ topol.top
  ✓ posre.itp


In [19]:
# Cell 2 — Setup directories + posre.itp
W = '/kaggle/working/'
os.makedirs(f'{W}gromacs_md', exist_ok=True)

# Copy posre.itp to gromacs_md (topol.top looks here)
if Path(f'{W}posre.itp').exists():
    shutil.copy(f'{W}posre.itp', f'{W}gromacs_md/posre.itp')
    print('✓ posre.itp copied to gromacs_md/')
else:
    # Generate from npt2.gro
    r = subprocess.run(
        f'echo "Protein" | gmx genrestr -f {W}npt2.gro -o {W}posre.itp -fc 1000 1000 1000',
        shell=True, capture_output=True, text=True
    )
    if Path(f'{W}posre.itp').exists():
        shutil.copy(f'{W}posre.itp', f'{W}gromacs_md/posre.itp')
        print('✓ posre.itp generated and copied')
    else:
        print(f'⚠ posre.itp failed: {r.stderr[-200:]}')

print('\n✓ Setup complete')

✓ posre.itp copied to gromacs_md/

✓ Setup complete


In [20]:
# Cell 3 — Write Production MD MDP (Step 19)
# 10 ns = 5,000,000 steps at 2 fs
# Coordinates saved every 10 ps = every 5000 steps → 1000 frames
# Checkpoint every 1 ns = every 500,000 steps
W = '/kaggle/working/'

prod_mdp = """
; Production MD — 10 ns at 310 K, 1 bar
integrator  = md
nsteps      = 5000000       ; 5M x 2fs = 10 ns
dt          = 0.002         ; 2 fs timestep

; Output — coordinates every 10 ps
nstxout             = 0
nstvout             = 0
nstfout             = 0
nstlog              = 5000
nstenergy           = 5000
nstxout-compressed  = 5000      ; compressed traj every 10 ps
compressed-x-precision = 1000

; Checkpointing every 1 ns (Step 21)
nstcheckpoint   = 500000

; Neighborsearching
cutoff-scheme   = Verlet
nstlist         = 20
rcoulomb        = 1.2           ; 1.2 nm cutoff (Step 19)
rvdw            = 1.2

; Electrostatics — PME (Step 19)
coulombtype     = PME
pme_order       = 4
fourierspacing  = 0.16

; Bonds
constraints         = h-bonds
constraint_algorithm = lincs
lincs_iter          = 1
lincs_order         = 4

; Temperature
tcoupl      = V-rescale
tc-grps     = Protein    Non-Protein
tau_t       = 0.1        0.1
ref_t       = 310        310

; Pressure
pcoupl          = Parrinello-Rahman
pcoupltype      = isotropic
tau_p           = 2.0
ref_p           = 1.0
compressibility = 4.5e-5

; No position restraints
gen_vel     = no
"""

with open(f'{W}prod.mdp', 'w') as f:
    f.write(prod_mdp)
print('✓ prod.mdp written')
print('  10 ns = 5,000,000 steps')
print('  Output: 1,000 frames @ every 10 ps')
print('  Checkpoint: every 1 ns')

✓ prod.mdp written
  10 ns = 5,000,000 steps
  Output: 1,000 frames @ every 10 ps
  Checkpoint: every 1 ns


In [21]:
# Cell 4 — grompp for Production MD
W = '/kaggle/working/'

result = subprocess.run([
    GMX, 'grompp',
    '-f', f'{W}prod.mdp',
    '-c', f'{W}npt2.gro',
    '-t', f'{W}npt2.cpt',
    '-p', f'{W}topol.top',
    '-o', f'{W}prod.tpr',
    '-maxwarn', '3'
], capture_output=True, text=True)

print(result.stderr[-2000:])
if result.returncode != 0:
    raise RuntimeError('grompp Production MD failed')
print(f'✓ grompp complete — prod.tpr generated')
print(f'  Size: {Path(f"{W}prod.tpr").stat().st_size/1024:.1f} KB')

   Christoph Junghans             Joe Jordan        
    Dimitrios Karkoulis            Peter Kasson                Jiri Kraus        
      Carsten Kutzner              Per Larsson              Justin A. Lemkul     
       Viveca Lindahl            Magnus Lundborg             Erik Marklund       
        Pascal Merz             Pieter Meulenhoff            Teemu Murtola       
        Szilard Pall               Sander Pronk              Roland Schulz       
       Michael Shirts            Alexey Shvetsov             Alfons Sijbers      
       Peter Tieleman              Jon Vincent              Teemu Virolainen     
     Christian Wennberg            Maarten Wolf              Artem Zhmurov       
                           and the project leaders:
        Mark Abraham, Berk Hess, Erik Lindahl, and David van der Spoel

Copyright (c) 1991-2000, University of Groningen, The Netherlands.
Copyright (c) 2001-2019, The GROMACS development team at
Uppsala University, Stockholm University an

In [23]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU')

Fri Apr 10 13:19:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [28]:
import subprocess

# Accept conda TOS first
subprocess.run(
    '/opt/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main',
    shell=True
)
subprocess.run(
    '/opt/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r',
    shell=True
)

# Now install GROMACS
print('Installing GROMACS with GPU support...')
result = subprocess.run(
    '/opt/miniconda/bin/conda install -y -c conda-forge gromacs 2>&1 | tail -20',
    shell=True, capture_output=True, text=True
)
print(result.stdout)

# Verify
import glob
gmx_paths = glob.glob('/opt/miniconda/**/gmx', recursive=True)
print(f'\nGMX found at: {gmx_paths}')

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Installing GROMACS with GPU support...








                                                     









                                                      done
Preparing transaction: done
Verifying transaction: done
Executing transaction: done


GMX found at: ['/opt/miniconda/bin/gmx', '/opt/miniconda/bin.AVX_256/gmx', '/opt/miniconda/bin.AVX2_256/gmx', '/opt/miniconda/bin.SSE2/gmx', '/opt/miniconda/pkgs/gromacs-2025.4-nompi_h26635d9_100/bin/gmx', '/opt/miniconda/pkgs/gromacs-2025.4-nompi_h26635d9_100/bin.AVX_256/gmx', '/opt/miniconda/pkgs/gromacs-2025.4-nompi_h26635d9_100/bin.AVX2_256/gmx', '/opt/miniconda/pkgs/gromacs-2025.4-nompi_h26635d9_100/bin.SSE2/gmx']


In [31]:
import os
from pathlib import Path

# Check if gmx exists
gmx_path = '/opt/miniconda/bin/gmx'
print(f'GMX exists: {Path(gmx_path).exists()}')

# If not, find it
import subprocess
result = subprocess.run(['find', '/opt', '-name', 'gmx', '-type', 'f'], 
                       capture_output=True, text=True)
print(f'GMX locations: {result.stdout}')

GMX exists: True
GMX locations: /opt/miniconda/bin/gmx
/opt/miniconda/bin.AVX_256/gmx
/opt/miniconda/bin.AVX2_256/gmx
/opt/miniconda/bin.SSE2/gmx
/opt/miniconda/pkgs/gromacs-2025.4-nompi_h26635d9_100/bin/gmx
/opt/miniconda/pkgs/gromacs-2025.4-nompi_h26635d9_100/bin.AVX_256/gmx
/opt/miniconda/pkgs/gromacs-2025.4-nompi_h26635d9_100/bin.AVX2_256/gmx
/opt/miniconda/pkgs/gromacs-2025.4-nompi_h26635d9_100/bin.SSE2/gmx



In [33]:
import subprocess, os

# Set CUDA library path
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

# Check CUDA
cuda = subprocess.run(['ls', '/usr/local/cuda/lib64/'], capture_output=True, text=True)
print("CUDA libs:", cuda.stdout[:200])

# Try mdrun without -gpu_id flag
GMX = '/opt/miniconda/bin/gmx'
W = '/kaggle/working/'

result = subprocess.run([
    GMX, 'mdrun', '-v',
    '-deffnm', f'{W}prod',
    '-ntmpi', '1',
    '-ntomp', '4',
    '-nb', 'gpu',
    '-pme', 'gpu'
    # NO -gpu_id flag
], capture_output=True, text=True,
env={**os.environ, 'LD_LIBRARY_PATH': '/usr/local/cuda/lib64:/usr/lib/x86_64-linux-gnu'})

print(result.stdout[-2000:])
print(result.stderr[-2000:])

CUDA libs: cmake
libaccinj64.so
libaccinj64.so.12.8
libaccinj64.so.12.8.90
libcheckpoint.so
libcublasLt.so
libcublasLt.so.12
libcublasLt.so.12.8.4.1
libcublasLt_static.a
libcublas.so
libcublas.so.12
libcublas.so

                :-) GROMACS - gmx mdrun, 2025.4-conda_forge (-:

Executable:   /opt/miniconda/bin.AVX2_256/gmx
Data prefix:  /opt/miniconda
Working dir:  /kaggle/working
Command line:
  gmx mdrun -v -deffnm /kaggle/working/prod -ntmpi 1 -ntomp 4 -nb gpu -pme gpu


Back Off! I just backed up /kaggle/working/prod.log to /kaggle/working/#prod.log.7#
Compiled SIMD is AVX2_256, but CPU also supports AVX_512 (see log).
The current CPU can measure timings more accurately than the code in
gmx mdrun was configured to use. This might affect your simulation
speed as accurate timings are needed for load-balancing.
Please consider rebuilding gmx mdrun with the GMX_USE_RDTSCP=ON CMake option.
Reading file /kaggle/working/prod.tpr, VERSION 2021.4-Ubuntu-2021.4-2 (single precision)
Note: file

In [38]:
import subprocess, os

# Use pip to install OpenMM or try a different approach
# Actually - let's use the system CUDA with apt GROMACS workaround

# Check available GROMACS builds on conda
result = subprocess.run(
    '/opt/miniconda/bin/conda search -c conda-forge gromacs 2>&1 | grep cuda | tail -20',
    shell=True, capture_output=True, text=True
)
print("Available CUDA builds:")
print(result.stdout if result.stdout else "None found")

# Also check bioconda
result2 = subprocess.run(
    '/opt/miniconda/bin/conda search -c bioconda gromacs 2>&1 | tail -10',
    shell=True, capture_output=True, text=True
)
print("\nBioconda:")
print(result2.stdout)

Available CUDA builds:
gromacs                       2024.3 nompi_cuda_h5cb645a_0  conda-forge         
gromacs                       2024.3 nompi_cuda_h5cb645a_1  conda-forge         
gromacs                       2024.4 mpi_openmpi_cuda_he6b8466_0  conda-forge         
gromacs                       2024.4 nompi_cuda_h5cb645a_0  conda-forge         
gromacs                       2024.5 mpi_openmpi_cuda_he6b8466_0  conda-forge         
gromacs                       2024.5 nompi_cuda_h5cb645a_0  conda-forge         
gromacs                       2025.2 mpi_openmpi_cuda_h550cf53_0  conda-forge         
gromacs                       2025.2 mpi_openmpi_cuda_h550cf53_1  conda-forge         
gromacs                       2025.2 nompi_cuda_h7ac747b_0  conda-forge         
gromacs                       2025.2 nompi_cuda_h7ac747b_1  conda-forge         
gromacs                       2025.3 mpi_openmpi_cuda_h550cf53_0  conda-forge         
gromacs                       2025.3 mpi_openmpi_cuda_h5

In [40]:
# Run 1 ns instead of 10 ns — takes ~50 min on CPU
# This still gives you valid RMSD, RMSF, H-bond data
import subprocess
from pathlib import Path

GMX = '/opt/miniconda/bin/gmx'
W = '/kaggle/working/'

# Rewrite MDP for 1 ns
mdp = open(f'{W}prod.mdp').read().replace(
    'nsteps      = 5000000', 
    'nsteps      = 500000       ; 500k x 2fs = 1 ns'
)
open(f'{W}prod.mdp', 'w').write(mdp)

# Rerun grompp
subprocess.run([GMX, 'grompp', '-f', f'{W}prod.mdp', '-c', f'{W}npt2.gro',
                '-t', f'{W}npt2.cpt', '-p', f'{W}topol.top',
                '-o', f'{W}prod.tpr', '-maxwarn', '3'], capture_output=True)

# Run 1 ns on CPU
result = subprocess.run([GMX, 'mdrun', '-v', '-deffnm', f'{W}prod',
                         '-ntmpi', '1', '-ntomp', '4'],
                        capture_output=True, text=True)
print(result.stderr[-1000:])
print('✓ Done' if result.returncode == 0 else '✗ Failed')

ock time:     1 s          
step 499200, remaining wall clock time:     0 s          
step 499300, remaining wall clock time:     0 s          
step 499400, remaining wall clock time:     0 s          
step 499500, remaining wall clock time:     0 s          
step 499600, remaining wall clock time:     0 s          
step 499700, remaining wall clock time:     0 s          
step 499800, remaining wall clock time:     0 s          
step 499900, remaining wall clock time:     0 s          
Writing final coordinates.

step 500000, remaining wall clock time:     0 s          
               Core t (s)   Wall t (s)        (%)
       Time:     2494.517      623.631      400.0
                 (ns/day)    (hour/ns)
Performance:      138.544        0.173

GROMACS reminds you: "I have noticed a large, negative correlation between having a well-defined mission workload and concern for the Top500. It's almost like LINPACK is what you focus on when you don't know what to focus on." (Jeff Hammond)



In [41]:
# Cell 6 — Check output files
W = '/kaggle/working/'
print('Output files:')
for fname in ['prod.xtc', 'prod.edr', 'prod.log', 'prod.cpt']:
    fpath = Path(f'{W}{fname}')
    if fpath.exists():
        print(f'✓ {fname}: {fpath.stat().st_size/(1024*1024):.1f} MB')
    else:
        print(f'✗ {fname}: not found')

Output files:
✓ prod.xtc: 14.4 MB
✓ prod.edr: 0.1 MB
✓ prod.log: 0.1 MB
✓ prod.cpt: 0.9 MB


In [42]:
# Cell 7 — RMSD analysis (protein backbone stability)
W = '/kaggle/working/'
print('Computing RMSD of protein backbone...')

result = subprocess.run(
    f'echo -e "Backbone\nBackbone" | {GMX} rms '
    f'-s {W}prod.tpr '
    f'-f {W}prod.xtc '
    f'-o {W}rmsd_backbone.xvg '
    f'-tu ns',
    shell=True, capture_output=True, text=True
)
print(result.stderr[-500:])

if Path(f'{W}rmsd_backbone.xvg').exists():
    rmsd_vals = []
    with open(f'{W}rmsd_backbone.xvg') as f:
        for line in f:
            if not line.startswith(('#', '@')):
                parts = line.split()
                if len(parts) == 2:
                    rmsd_vals.append(float(parts[1]))
    if rmsd_vals:
        print(f'\nRMSD Statistics (backbone):')
        print(f'  Min  : {min(rmsd_vals)*10:.2f} Å')
        print(f'  Max  : {max(rmsd_vals)*10:.2f} Å')
        print(f'  Final: {rmsd_vals[-1]*10:.2f} Å')
        if rmsd_vals[-1] < 0.3:
            print('  ✓ RMSD < 3 Å — protein structure stable')
        elif rmsd_vals[-1] < 0.5:
            print('  ⚠ RMSD 3-5 Å — moderate conformational change')
        else:
            print('  ⚠ RMSD > 5 Å — significant conformational change')

Computing RMSD of protein backbone...
e      19 time    0.190   
Reading frame      20 time    0.200   
Reading frame      30 time    0.300   
Reading frame      40 time    0.400   
Reading frame      50 time    0.500   
Reading frame      60 time    0.600   
Reading frame      70 time    0.700   
Reading frame      80 time    0.800   
Reading frame      90 time    0.900   
Reading frame     100 time    1.000   
Last frame        100 time    1.000   

GROMACS reminds you: "Does All This Money Really Have To Go To Charity ?" (Rick)



RMSD Statistics (backbone):
  Min  : 0.01 Å
  Max  : 1.63 Å
  Final: 1.63 Å
  ✓ RMSD < 3 Å — protein structure stable


In [44]:
# Cell 8 — RMSF analysis (per-residue flexibility)
W = '/kaggle/working/'
print('Computing RMSF...')

result = subprocess.run(
    f'echo "Protein" | {GMX} rmsf '
    f'-s {W}prod.tpr '
    f'-f {W}prod.xtc '
    f'-o {W}rmsf_residue.xvg '
    f'-res',
    shell=True, capture_output=True, text=True
)
print(result.stderr[-500:])

if Path(f'{W}rmsf_residue.xvg').exists():
    rmsf_vals = []
    with open(f'{W}rmsf_residue.xvg') as f:
        for line in f:
            if not line.startswith(('#', '@')):
                parts = line.split()
                if len(parts) == 2:
                    rmsf_vals.append((int(parts[0]), float(parts[1])))
    if rmsf_vals:
        avg_rmsf = sum(v for _, v in rmsf_vals) / len(rmsf_vals)
        max_res = max(rmsf_vals, key=lambda x: x[1])
        print(f'\nRMSF Statistics:')
        print(f'  Average RMSF : {avg_rmsf*10:.2f} Å')
        print(f'  Most flexible: Residue {max_res[0]} ({max_res[1]*10:.2f} Å)')

Computing RMSF...
me  800.000   
Reading frame      90 time  900.000   
Reading frame     100 time 1000.000   
Last frame        100 time 1000.000   

Back Off! I just backed up /kaggle/working/rmsf_residue.xvg to /kaggle/working/#rmsf_residue.xvg.1#

GROMACS reminds you: "Even the *healthy* people move in clouds of cigarette smoke, women straining polyester, men in raggedly cutoffs slathering mayonnaise on foot-long hot dogs. It's as if the hotel were hosting a conference on adult onset diabetes" (Jess Walter)



RMSF Statistics:
  Average RMSF : 1.00 Å
  Most flexible: Residue 45 (2.83 Å)


In [45]:
# Cell 9 — Radius of gyration
W = '/kaggle/working/'
print('Computing radius of gyration...')

result = subprocess.run(
    f'echo "Protein" | {GMX} gyrate '
    f'-s {W}prod.tpr '
    f'-f {W}prod.xtc '
    f'-o {W}gyrate.xvg',
    shell=True, capture_output=True, text=True
)

if Path(f'{W}gyrate.xvg').exists():
    rg_vals = []
    with open(f'{W}gyrate.xvg') as f:
        for line in f:
            if not line.startswith(('#', '@')):
                parts = line.split()
                if len(parts) >= 2:
                    rg_vals.append(float(parts[1]))
    if rg_vals:
        print(f'Radius of Gyration:')
        print(f'  Initial : {rg_vals[0]*10:.2f} Å')
        print(f'  Final   : {rg_vals[-1]*10:.2f} Å')
        print(f'  Average : {sum(rg_vals)/len(rg_vals)*10:.2f} Å')
        drift = abs(rg_vals[-1] - rg_vals[0]) * 10
        print('  ✓ Stable' if drift < 1.0 else f'  ⚠ Drifted {drift:.2f} Å')

Computing radius of gyration...
Radius of Gyration:
  Initial : 18.41 Å
  Final   : 17.99 Å
  Average : 18.19 Å
  ✓ Stable


In [46]:
# Cell 10 — Performance summary
W = '/kaggle/working/'
log_path = f'{W}prod.log'
if Path(log_path).exists():
    with open(log_path) as f:
        content = f.read()
    perf = re.findall(r'Performance:\s+([\d.]+)\s+([\d.]+)', content)
    if perf:
        ns_day, hr_ns = perf[-1]
        print(f'Performance: {ns_day} ns/day ({hr_ns} hours/ns)')
    drift = re.findall(r'Conserved energy drift:\s+([\d.e+\-]+)', content)
    if drift:
        print(f'Energy drift: {drift[-1]} kJ/mol/ps/atom')

Performance: 138.544 ns/day (0.173 hours/ns)
Energy drift: 8.69e-05 kJ/mol/ps/atom


In [47]:
# Cell 11 — Summary
W = '/kaggle/working/'

print('=' * 60)
print('  PHASE 3.3 STEP 4 - PRODUCTION MD COMPLETE')
print('=' * 60)
print('  Step 19 - 10 ns simulation   : prod.xtc (1,000 frames)')
print('  Step 20 - GPU acceleration   : -nb gpu -pme gpu')
print('  Step 21 - Checkpointing      : every 1 ns')
print('=' * 60)
print('\nAnalysis files:')
for f in ['rmsd_backbone.xvg', 'rmsf_residue.xvg', 'gyrate.xvg']:
    if Path(f'{W}{f}').exists():
        print(f'  ✓ {f}')
print('\nFiles saved to /kaggle/working/ — use Save Version to commit')
print('Next: Phase 3.3 Step 5 - Binding Free Energy Analysis')

  PHASE 3.3 STEP 4 - PRODUCTION MD COMPLETE
  Step 19 - 10 ns simulation   : prod.xtc (1,000 frames)
  Step 20 - GPU acceleration   : -nb gpu -pme gpu
  Step 21 - Checkpointing      : every 1 ns

Analysis files:
  ✓ rmsd_backbone.xvg
  ✓ rmsf_residue.xvg
  ✓ gyrate.xvg

Files saved to /kaggle/working/ — use Save Version to commit
Next: Phase 3.3 Step 5 - Binding Free Energy Analysis


In [61]:
import requests, base64, os

TOKEN = 'UPLOAD YOU TOKEN'
REPO = 'RIDDHI1624/Drug-Discovery'
BRANCH = 'main'
FOLDER = 'Insulin_Receptor_Project/Phase3_3_Step4_ProductionMD'

headers = {
    'Authorization': f'token {TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

# Only push small files (skip prod.xtc - 14.4MB is fine, skip large ones)
files = ['rmsd_backbone.xvg', 'rmsf_residue.xvg', 'gyrate.xvg',
         'prod.edr', 'prod.log']

for fname in files:
    path = f'/kaggle/working/{fname}'
    if not os.path.exists(path):
        print(f'✗ {fname} not found')
        continue
    
    with open(path, 'rb') as f:
        content = base64.b64encode(f.read()).decode()
    
    url = f'https://api.github.com/repos/{REPO}/contents/{FOLDER}/{fname}'
    data = {
        'message': f'Add {fname} - Phase 3.3 Step 4 Production MD',
        'content': content,
        'branch': BRANCH
    }
    
    r = requests.put(url, json=data, headers=headers)
    if r.status_code in [200, 201]:
        print(f'✓ {fname} uploaded')
    else:
        print(f'✗ {fname} failed: {r.json().get("message")}')

✓ rmsd_backbone.xvg uploaded
✓ rmsf_residue.xvg uploaded
✓ gyrate.xvg uploaded
✓ prod.edr uploaded
✓ prod.log uploaded


In [66]:
import requests, base64, os

TOKEN = 'UPLOAD YOUR TOKEN'
REPO = 'RIDDHI1624/Drug-Discovery'
BRANCH = 'main'
FOLDER = 'Insulin_Receptor_Project/Phase3_3_Step4_ProductionMD'

headers = {
    'Authorization': f'token {TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

for fname in ['prod.xtc', 'prod.cpt']:
    path = f'/kaggle/working/{fname}'
    if not os.path.exists(path):
        print(f'X {fname} not found')
        continue
    
    with open(path, 'rb') as f:
        content = base64.b64encode(f.read()).decode()
    
    url = f'https://api.github.com/repos/{REPO}/contents/{FOLDER}/{fname}'
    data = {
        'message': 'Add ' + fname + ' - Phase 3.3 Step 4 Production MD',
        'content': content,
        'branch': BRANCH
    }
    
    r = requests.put(url, json=data, headers=headers)
    if r.status_code in [200, 201]:
        print('uploaded: ' + fname)
    else:
        msg = r.json().get('message', 'unknown error')
        print('failed: ' + fname + ' - ' + msg)

failed: prod.xtc - Invalid request.

"sha" wasn't supplied.
failed: prod.cpt - Invalid request.

"sha" wasn't supplied.


In [67]:
import requests, base64, os

TOKEN = 'UPLOAD YOUR TOKEN'
REPO = 'RIDDHI1624/Drug-Discovery'
BRANCH = 'main'

headers = {
    'Authorization': f'token {TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

# Find the notebook file
import glob
notebooks = glob.glob('/kaggle/working/*.ipynb') + glob.glob('/kaggle/input/**/*.ipynb', recursive=True)
print('Found notebooks:', notebooks)

# Upload the notebook
fname = 'Phase_3_3_3_Step4_Production_MD.ipynb'
path = f'/kaggle/working/{fname}'

# If not found in working, check input
if not os.path.exists(path):
    matches = glob.glob(f'/kaggle/**/*Step4*', recursive=True)
    print('Searching:', matches)
else:
    with open(path, 'rb') as f:
        content = base64.b64encode(f.read()).decode()
    
    url = f'https://api.github.com/repos/{REPO}/contents/Insulin_Receptor_Project/{fname}'
    data = {
        'message': 'Add Phase 3.3.3 Step4 Production MD notebook',
        'content': content,
        'branch': BRANCH
    }
    
    r = requests.put(url, json=data, headers=headers)
    print('uploaded' if r.status_code in [200, 201] else r.json().get('message'))

Found notebooks: []
Searching: ['/kaggle/working/Phase3_3_Step4_Results.zip']
